# Creazione Dataset, Feature, Modelli

In [ ]:

# ============================================================
# Tesi: Generazione e Analisi Serie Temporali Glucosio
# Script Unificato: V1 (15d), V2 (30d), V3 (60/90d)
# ============================================================

# 1. IMPORT & SETUP
import os
import random
import re
import glob
import numpy as np
import pandas as pd
import joblib
import networkx as nx
from collections import defaultdict, Counter
from tqdm.auto import tqdm
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

# Configurazione Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Running on: {DEVICE}")

# --- CONFIGURAZIONE PATH ---
BASE_PATH = "drive/MyDrive/tesi" # Modifica se necessario
CSV_PATH  = os.path.join(BASE_PATH, "intervalli_glucosio.csv")
DATASET_DIR = os.path.join(BASE_PATH, "datasets")
FEATURE_DIR = os.path.join(BASE_PATH, "features_opt")
MODEL_DIR   = os.path.join(BASE_PATH, "models_opt")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,   exist_ok=True)

# Mappature
TARGET_TO_INT = {"rosso": 0, "giallo": 1, "verde": 2}
STATE_ID = {"extremely_low": 0, "low": 1, "normal": 2, "high": 3, "extremely_high": 4}
NUM_STATES = len(STATE_ID)

# ============================================================
# 2. COSTRUZIONE GRAFO (COMUNE A TUTTE LE VERSIONI)
# ============================================================
print(f"📊 Caricamento Grafo da: {CSV_PATH}")

if os.path.exists(CSV_PATH):
    interval_labeling = pd.read_csv(CSV_PATH, delimiter=',')
    df_csv = pd.DataFrame(interval_labeling)
    df_csv["start_time"] = pd.to_datetime(df_csv["start_time"])
    df_csv["end_time"] = pd.to_datetime(df_csv["end_time"])
    df_csv["duration"] = ((df_csv["end_time"] - df_csv["start_time"]).dt.total_seconds() / 60).round(2)

    arr = []
    for index, row in df_csv.iterrows():
        label = str(row['label']).strip().lower()
        arr.append((label, row['duration']))

    G = nx.DiGraph()
    for i in range(len(arr) - 1):
        src, dur_src = arr[i]
        dst, dur_dst = arr[i + 1]
        if src not in STATE_ID or dst not in STATE_ID: continue
        if G.has_edge(src, dst):
            G[src][dst]["count"] += 1
            G[src][dst]["durations"].append(dur_dst)
        else:
            G.add_edge(src, dst, count=1, durations=[dur_dst])

    for u, v, data in G.edges(data=True):
        data["mean_duration"] = np.mean(data["durations"])
        data["std_duration"] = np.std(data["durations"]) if len(data["durations"]) > 1 else 1.0
    print(f"✔ Grafo Costruito.")
else:
    print("❌ ERRORE: CSV non trovato! Fallback...")
    G = nx.DiGraph()
    for s in STATE_ID: G.add_edge(s, s, mean_duration=30, count=1)

# ============================================================
# 3. HELPER FUNCTIONS (ANALISI & INTEGRITÀ)
# ============================================================
def normalize_target(target):
    return re.sub(r'[^\w]', '', str(target)).lower()

def analyze_simulate_sequence_compact(seq, analize_day=None):
    total_minutes = sum(item[1] for item in seq)
    if analize_day is None: analize_day = max(1, round(total_minutes / 1440))
    max_minutes = analize_day * 1440
    gl_values = []
    elapsed = 0
    for (state, dur, glucose) in seq:
        for _ in range(int(dur)):
            if elapsed >= max_minutes: break
            gl_values.append(glucose)
            elapsed += 1
        if elapsed >= max_minutes: break
    gl_values = np.array(gl_values)
    if len(gl_values) < 10: return "🟢 Verde", {}

    TBR1 = np.mean(gl_values < 70)*100
    TIR  = np.mean((gl_values >= 70)&(gl_values <= 180))*100
    TAR1 = np.mean(gl_values > 180)*100
    GV   = np.std(gl_values)

    status = "🟢 Verde"
    if (TIR < 55) or (TBR1 > 10) or (TAR1 > 40) or (GV > 70): status = "🔴 Rosso"
    elif (TIR < 70) or (TBR1 > 4) or (TAR1 > 25) or (GV > 50): status = "🟡 Giallo"
    return status, {}

def verify_dataset_integrity(df, days, expected_count=None):
    print(f"\n🕵️‍♂️ REPORT INTEGRITÀ ({days}d)")

    current_count = len(df)
    print(f"   📊 Righe Totali: {current_count}")
    if expected_count and current_count != expected_count:
        diff = expected_count - current_count
        print(f"   ⚠️ WARNING: Mancano {diff} righe al target di {expected_count}!")
        if current_count < (expected_count * 0.9): return False

    unique_targets = df["target"].unique()
    invalid = [t for t in unique_targets if t not in TARGET_TO_INT]
    if invalid:
        print(f"   ❌ FATAL: Label invalide: {invalid}")
        return False

    sample = df.sample(min(len(df), 50), random_state=42)
    errors = 0
    for _, row in sample.iterrows():
        recalc, _ = analyze_simulate_sequence_compact(row['sequence'])
        if normalize_target(recalc) != row['target']:
            errors += 1
    if errors > 5:
        print(f"   ❌ CRITICO: Trovati {errors}/50 mismatch clinici!")
        return False

    print("   ✅ DATASET OK.")
    return True

# ============================================================
# 4. LE 3 VERSIONI DEGLI ALGORITMI DI SIMULAZIONE
# ============================================================

# --- V1: Short Term Logic (Per 15 giorni) ---
# Caratteristica: Più stocastica, meno forzature. Si affida al grafo.
def simulate_v1_short(G, total_minutes, target_color, ranges):
    seq = []
    state = "normal"
    elapsed = 0

    # Pesi standard (poco aggressivi)
    if target_color == "verde": base_probs = {"normal": 0.7, "high": 0.15, "low": 0.15, "extreme": 0.0}
    elif target_color == "giallo": base_probs = {"normal": 0.4, "high": 0.3, "low": 0.2, "extreme": 0.1}
    else: base_probs = {"normal": 0.2, "high": 0.4, "low": 0.2, "extreme": 0.2}

    while elapsed < total_minutes:
        successors = list(G.successors(state))
        if not successors: state = "normal"; successors = list(G.successors(state))

        weights = []
        for s in successors:
            w = base_probs.get("normal", 0.5)
            if s == "normal": w = base_probs["normal"]
            elif s in ["high", "low"]: w = base_probs["high"]
            else: w = base_probs["extreme"]
            weights.append(w)

        next_state = random.choices(successors, weights=weights, k=1)[0]

        # Durata standard dal grafo (nessun moltiplicatore)
        dur = G[state][next_state].get("mean_duration", 30) * np.random.uniform(0.8, 1.2)
        dur = max(5, min(dur, 180))

        gmin, gmax = ranges.get(next_state, (90,140))
        val = np.random.uniform(gmin, gmax) + np.random.normal(0, 5) # Poco rumore

        seq.append((next_state, int(dur), round(val, 1)))
        state = next_state
        elapsed += dur

    return seq

# --- V2: Mid Term Logic (Per 30 giorni) ---
# Caratteristica: Introdotta logica guidata per evitare deriva dei target
def simulate_v2_mid(G, total_minutes, target_color, ranges):
    seq = []
    state = "normal"
    elapsed = 0
    counts = defaultdict(int)

    # Pesi mediamente sbilanciati
    if target_color == "verde": prob_weights = {"normal": 10, "high": 1, "low": 1, "extreme": 0}
    elif target_color == "giallo": prob_weights = {"normal": 3, "high": 3, "low": 2, "extreme": 1}
    else: prob_weights = {"normal": 1, "high": 5, "low": 2, "extreme": 2} # Rosso inizia a spingere

    while elapsed < total_minutes:
        successors = list(G.successors(state))
        if not successors: state = "normal"; successors = list(G.successors(state))

        weights = []
        for s in successors:
            w = 1.0
            if s == "normal": w = prob_weights["normal"]
            elif s in ["high", "low"]: w = prob_weights["high"] if s=="high" else prob_weights["low"]
            else: w = prob_weights["extreme"]
            # Penalità ripetizione
            if counts[s] > (total_minutes/30): w *= 0.8
            weights.append(w)

        next_state = random.choices(successors, weights=weights, k=1)[0]

        # Durata leggermente aumentata per patologie
        mult = 1.2 if target_color == "rosso" else 1.0
        dur = G[state][next_state].get("mean_duration", 30) * mult * np.random.uniform(0.7, 1.3)
        dur = max(5, min(dur, 240))

        gmin, gmax = ranges.get(next_state, (90,140))
        val = np.random.uniform(gmin, gmax) + np.random.normal(0, 10) # Rumore medio

        seq.append((next_state, int(dur), round(val, 1)))
        counts[next_state] += 1
        state = next_state
        elapsed += dur

    return seq

# --- V3: Long Term / Advanced Logic (Per 60/90 giorni) ---
# Caratteristica: Driver Fisiologici, Sovrapposizione Classi, Chunking Ready
def simulate_v3_advanced(G, total_minutes, target_color, ranges):
    seq = []

    # Pesi Clinici & Sovrapposizione (Fix F1=1.0)
    if target_color == "verde":
        prob_weights = {"normal": 15, "high": 2, "low": 2, "extreme": 0}
        duration_multiplier = 1.0; noise_std = 8
    elif target_color == "giallo":
        prob_weights = {"normal": 6, "high": 3, "low": 2, "extreme": 1}
        duration_multiplier = 1.1; noise_std = 12
    elif target_color == "rosso":
        # Torna in normal per confondere il modello (realismo)
        prob_weights = {"normal": 2, "high": 5, "low": 2, "extreme": 3}
        duration_multiplier = 1.3; noise_std = 18

    # Start point coerente
    if target_color == "verde": start_nodes = ["normal"]
    elif target_color == "rosso": start_nodes = ["high", "extremely_high"]
    else: start_nodes = ["high", "normal"]
    state = random.choice(start_nodes)

    elapsed = 0
    counts = defaultdict(int)

    while elapsed < total_minutes:
        successors = list(G.successors(state))
        if not successors: state = random.choice(list(G.nodes)); successors = list(G.successors(state))

        weights = []
        for s in successors:
            w = 1.0
            if s == "normal": w = prob_weights["normal"]
            elif s == "high": w = prob_weights["high"]
            elif s == "low":  w = prob_weights["low"]
            else: w = prob_weights["extreme"]
            if counts[s] > (total_minutes / 60): w *= 0.5
            weights.append(w)

        next_state = random.choices(successors, weights=weights, k=1)[0]

        # Durata Fisiologica Variabile
        base_dur = G[state][next_state].get("mean_duration", 30)
        if target_color == "rosso" and next_state in ["high", "extremely_high"]:
            current_dur = base_dur * duration_multiplier * np.random.uniform(0.8, 1.4)
        else:
            current_dur = base_dur * np.random.uniform(0.6, 1.4)
        current_dur = max(5, min(current_dur, 300))

        gmin, gmax = ranges.get(next_state, (90,140))
        val = np.random.uniform(gmin, gmax) + np.random.normal(0, noise_std)
        val = np.clip(val, 20, 600)

        seq.append((next_state, int(current_dur), round(val, 1)))
        counts[next_state] += 1
        state = next_state
        elapsed += current_dur

    return seq

# --- WRAPPER UNIFICATO ---
def simulate_selector(G, days, target_req, forced_length=None):
    total_minutes = forced_length if forced_length else days * 1440

    # Definizione Range Glucosio (Comune con piccole variazioni)
    ranges = {
        "extremely_low": (40, 60), "low": (60, 75),
        "normal": (80, 160),
        "high": (170, 240), "extremely_high": (250, 400)
    }

    # SELEZIONE STRATEGIA
    if days <= 15:
        # V1: Semplice
        seq = simulate_v1_short(G, total_minutes, target_req, ranges)
    elif days <= 30:
        # V2: Guidata
        seq = simulate_v2_mid(G, total_minutes, target_req, ranges)
    else:
        # V3: Avanzata (60/90+)
        seq = simulate_v3_advanced(G, total_minutes, target_req, ranges)

    true_color, _ = analyze_simulate_sequence_compact(seq)

    # Calcolo avg rapido
    tot_val = sum(v*d for _,d,v in seq)
    tot_time = sum(d for _,d,_ in seq)
    avg_gluc = tot_val/tot_time if tot_time>0 else 100

    return seq, true_color, 0, avg_gluc

# ============================================================
# 5. GENERAZIONE ROBUSTA (SELF-HEALING + CHUNKED)
# ============================================================
def generate_and_save_chunked(G, N_dict, cache_dir, chunk_size=5000):
    if not os.path.exists(cache_dir): os.makedirs(cache_dir)

    for days, N in N_dict.items():
        final_path = os.path.join(cache_dir, f"dataset_{days}d.pkl")
        if os.path.exists(final_path):
            print(f"   ✔ Dataset {days}d completo esistente. Skip.")
            continue

        print(f"   ⚙ Gestione dataset {days}d (Target: {N})...")

        # A. Migrazione Vecchi Checkpoint
        old_checkpoint = os.path.join(cache_dir, f"dataset_{days}d_temp_checkpoint.pkl")
        if os.path.exists(old_checkpoint):
            try:
                legacy_data = joblib.load(old_checkpoint)
                if isinstance(legacy_data, pd.DataFrame): legacy_data = legacy_data.to_dict('records')
                new_part_path = os.path.join(cache_dir, f"temp_{days}d_part_legacy.pkl")
                joblib.dump(legacy_data, new_part_path)
                print(f"      ✔ Migrato vecchio checkpoint.")
                os.remove(old_checkpoint)
            except: os.remove(old_checkpoint) if os.path.exists(old_checkpoint) else None

        # B. Validazione Chunk Esistenti
        chunk_pattern = os.path.join(cache_dir, f"temp_{days}d_part_*.pkl")
        existing_chunks = glob.glob(chunk_pattern)
        needed = {"verde": int(N * 0.6), "giallo": int(N * 0.3), "rosso": int(N * 0.1)}

        if existing_chunks:
            print(f"      ⟳ Verifica integrità {len(existing_chunks)} pezzi esistenti...")
            for chunk_file in existing_chunks:
                try:
                    chunk_data = joblib.load(chunk_file)
                    for row in chunk_data:
                        t = row['target']
                        if t in needed:
                            needed[t] -= 1
                            if needed[t] < 0: needed[t] = 0
                except:
                    print(f"      ⚠ CORROTTO: {chunk_file} -> Eliminato.")
                    try: os.remove(chunk_file)
                    except: pass
            print(f"      ⟳ Mancano ancora: {needed}")

        # C. Generazione
        if sum(needed.values()) <= 0:
            print("      Quota raggiunta con i chunk esistenti.")
        else:
            current_buffer = []
            chunk_idx = len(existing_chunks) + 50
            pbar = tqdm(total=sum(needed.values()), desc=f"Gen {days}d")

            while sum(needed.values()) > 0:
                active_needs = {k: v for k, v in needed.items() if v > 0}
                if not active_needs: break
                target_req = max(active_needs, key=active_needs.get)

                # CHIAMA IL SELETTORE DI STRATEGIA
                seq, target_raw, _, _ = simulate_selector(G, days, target_req)
                t_clean = normalize_target(target_raw)

                if t_clean in needed and needed[t_clean] > 0:
                    current_buffer.append({"sequence": seq, "target": t_clean})
                    needed[t_clean] -= 1
                    pbar.update(1)

                if len(current_buffer) >= chunk_size:
                    save_path = os.path.join(cache_dir, f"temp_{days}d_part_{chunk_idx}.pkl")
                    joblib.dump(current_buffer, save_path)
                    chunk_idx += 1
                    current_buffer = []

            pbar.close()
            if current_buffer:
                joblib.dump(current_buffer, os.path.join(cache_dir, f"temp_{days}d_part_{chunk_idx}.pkl"))

        # D. Merge Finale
        print(f"   🔗 Unione finale dataset {days}d...")
        all_chunks = glob.glob(chunk_pattern)
        full_dataset = []
        for cf in tqdm(all_chunks, desc="Merging"):
            try: full_dataset.extend(joblib.load(cf))
            except: pass

        df = pd.DataFrame(full_dataset)
        df = df.sample(frac=1).reset_index(drop=True)
        joblib.dump(df, final_path)

        for cf in all_chunks:
            try: os.remove(cf)
            except: pass
        print(f"   ✔ Dataset {days}d completato.\n")

# ============================================================
# 6. FEATURE & MODEL
# ============================================================
def transform_glucose_risk(g):
    g_clipped = np.clip(g, 20, 600)
    return 1.509 * (np.log(g_clipped)**1.084 - 5.381)

def build_compressed_features_v2(run_list):
    T = len(run_list)
    if T == 0: return np.zeros((1, 20))
    states = np.array([STATE_ID.get(s, 2) for s,_,_ in run_list], dtype=int)
    durs = np.array([d for _,d,_ in run_list], dtype=float)
    gs = np.array([g for *_,g in run_list], dtype=float)

    state_oh = np.eye(NUM_STATES)[states]
    prev_s = np.roll(states, 1); prev_s[0]=states[0]; prev_oh = np.eye(NUM_STATES)[prev_s]
    dur_norm = np.clip(durs / 240.0, 0, 1)
    log_dur = np.log1p(durs)
    g_delta = gs - np.roll(gs, 1); g_delta[0]=0
    with np.errstate(divide='ignore', invalid='ignore'): roc = np.nan_to_num(g_delta/durs)
    risk = transform_glucose_risk(gs)
    load = (gs - 100)*(durs/60.0)
    zsc = (gs-140)/50.0
    roll_risk = np.convolve(risk, np.ones(5)/5, mode='same')
    feats = np.column_stack([state_oh, prev_oh, dur_norm, log_dur, zsc, g_delta, roc, risk, roll_risk, load])
    return np.nan_to_num(feats).astype(np.float32)

class AttentionBlock(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        self.att = nn.Sequential(nn.Linear(h_dim, h_dim//2), nn.Tanh(), nn.Linear(h_dim//2, 1))
    def forward(self, x, lens):
        sc = self.att(x)
        mask = torch.zeros_like(sc, dtype=torch.bool)
        for i, l in enumerate(lens): mask[i, :l, :] = 1
        sc = sc.masked_fill(~mask, -float('inf'))
        return torch.sum(torch.softmax(sc, dim=1) * x, dim=1)

class LSTMAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attn = AttentionBlock(hidden_dim*2)
        self.ln = nn.LayerNorm(hidden_dim*2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, num_classes))
    def forward(self, x, l):
        packed = nn.utils.rnn.pack_padded_sequence(x, l.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        unpacked, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.fc(self.ln(self.attn(unpacked, l)))

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

# ============================================================
# 7. MAIN EXECUTION
# ============================================================
# Configura qui quali dataset vuoi creare/processare
DAYS_CONFIG = { 90: 50000 } # Esempio, puoi aggiungere 15, 30

print("\n=== STEP 1: GENERAZIONE DATASET UNIFICATA ===")
generate_and_save_chunked(G, DAYS_CONFIG, DATASET_DIR, chunk_size=5000)

print("\n=== STEP 1.5: VERIFICA INTEGRITÀ ===")
all_datasets_valid = True
valid_days_list = []

for days, target_n in DAYS_CONFIG.items():
    path = os.path.join(DATASET_DIR, f"dataset_{days}d.pkl")
    if not os.path.exists(path):
        all_datasets_valid = False; continue

    try:
        print(f"   🔎 Controllo dataset {days}d...")
        df_check = joblib.load(path)
        if verify_dataset_integrity(df_check, days, expected_count=target_n):
            valid_days_list.append(days)
        else: all_datasets_valid = False
        del df_check
    except: all_datasets_valid = False

if not all_datasets_valid:
    print("\n⛔ STOP: Dataset non validi.")
else:
    print("\n✅ Tutti i dataset sono validi. Procedo al training.")
    print("\n=== STEP 2: TRAINING MODEL ===")

    for days in valid_days_list:
        print(f"\n🌀 PROCESSING {days}d")
        df = joblib.load(os.path.join(DATASET_DIR, f"dataset_{days}d.pkl"))
        ft_path = os.path.join(FEATURE_DIR, f"features_opt_{days}d.pkl")

        if os.path.exists(ft_path):
            print("   ✔ Features caricate.")
            X = joblib.load(ft_path)
        else:
            print("   ⚙ Calcolo Features...")
            X = [build_compressed_features_v2(s) for s in tqdm(df["sequence"])]
            joblib.dump(X, ft_path, compress=3)

        if np.isnan(np.sum(X[0])): X = [np.nan_to_num(x) for x in X]
        y = [TARGET_TO_INT[t] for t in df["target"]]
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        weights = [1./np.bincount(y_tr)[label] for label in y_tr]
        sampler = WeightedRandomSampler(weights, len(weights))

        def collate(b):
            x = nn.utils.rnn.pad_sequence([torch.tensor(i[0]) for i in b], batch_first=True)
            l = torch.tensor([len(i[0]) for i in b])
            y = torch.tensor([i[1] for i in b])
            return x, l, y

        dl_tr = DataLoader(list(zip(X_tr, y_tr)), batch_size=64, sampler=sampler, collate_fn=collate)
        dl_te = DataLoader(list(zip(X_te, y_te)), batch_size=64, collate_fn=collate)

        model = LSTMAttentionClassifier(input_dim=X[0].shape[1]).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = FocalLoss(gamma=2.0)

        print(f"   ⚡ Training {days}d...")
        best_f1 = 0

        for ep in range(15):
            model.train()
            l_sum = 0
            for x, l, y_b in dl_tr:
                optimizer.zero_grad()
                out = model(x.to(DEVICE), l.to(DEVICE))
                loss = criterion(out, y_b.to(DEVICE))
                loss.backward(); optimizer.step()
                l_sum += loss.item()

            model.eval()
            preds, trues = [], []; v_sum = 0
            with torch.no_grad():
                for x, l, y_b in dl_te:
                    out = model(x.to(DEVICE), l.to(DEVICE))
                    v_sum += criterion(out, y_b.to(DEVICE)).item()
                    preds.extend(out.argmax(1).cpu().tolist())
                    trues.extend(y_b.cpu().tolist())

            f1 = f1_score(trues, preds, average="macro")
            acc = accuracy_score(trues, preds)

            s_msg = ""
            if f1 > best_f1:
                best_f1 = f1
                torch.save(model.state_dict(), os.path.join(MODEL_DIR, f"model_{days}d.pth"))
                s_msg = "💾 Saved"

            print(f"     Ep {ep+1:02d}: TrL={l_sum/len(dl_tr):.3f} | VaL={v_sum/len(dl_te):.3f} | Acc={acc*100:.1f}% | F1={f1:.3f} {s_msg}")

        print(f"   🎯 FINAL REPORT {days}d:")
        print(classification_report(trues, preds, target_names=["ROSSO", "GIALLO", "VERDE"]))

# Analisi modelli

In [ ]:
import os
import joblib
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.auto import tqdm

# --- CONFIGURAZIONE ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_PATH = "drive/MyDrive/tesi" # O "." se sei in locale
DATASET_DIR = os.path.join(BASE_PATH, "datasets")
FEATURE_DIR = os.path.join(BASE_PATH, "features_opt")
MODEL_DIR   = os.path.join(BASE_PATH, "models_opt")

# Mappature
TARGET_TO_INT = {"rosso": 0, "giallo": 1, "verde": 2}
INT_TO_TARGET = {0: "ROSSO", 1: "GIALLO", 2: "VERDE"}
STATE_ID = {"extremely_low": 0, "low": 1, "normal": 2, "high": 3, "extremely_high": 4}
NUM_STATES = len(STATE_ID)

print(f"🚀 Avvio Valutazione su: {DEVICE}")

# --- RIDEFINIZIONE CLASSI (Necessaria per caricare i modelli) ---
class AttentionBlock(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        self.att = nn.Sequential(nn.Linear(h_dim, h_dim//2), nn.Tanh(), nn.Linear(h_dim//2, 1))
    def forward(self, x, lens):
        sc = self.att(x)
        mask = torch.zeros_like(sc, dtype=torch.bool)
        for i, l in enumerate(lens): mask[i, :l, :] = 1
        sc = sc.masked_fill(~mask, -float('inf'))
        return torch.sum(torch.softmax(sc, dim=1) * x, dim=1)

class LSTMAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attn = AttentionBlock(hidden_dim*2)
        self.ln = nn.LayerNorm(hidden_dim*2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, num_classes))
    def forward(self, x, l):
        packed = nn.utils.rnn.pack_padded_sequence(x, l.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        unpacked, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.fc(self.ln(self.attn(unpacked, l)))

def collate(b):
    x = nn.utils.rnn.pad_sequence([torch.tensor(i[0]) for i in b], batch_first=True)
    l = torch.tensor([len(i[0]) for i in b])
    y = torch.tensor([i[1] for i in b])
    return x, l, y

# --- FUNZIONE DI VALUTAZIONE ---
def evaluate_model(days):
    print(f"\n{'='*60}")
    print(f"📊 VALUTAZIONE MODELLO: {days} GIORNI")
    print(f"{'='*60}")

    # 1. Check File
    d_path = os.path.join(DATASET_DIR, f"dataset_{days}d.pkl")
    f_path = os.path.join(FEATURE_DIR, f"features_opt_{days}d.pkl")
    m_path = os.path.join(MODEL_DIR, f"model_{days}d.pth")

    if not os.path.exists(d_path) or not os.path.exists(m_path):
        print(f"⚠ Saltato: Mancano i dati o il modello per {days}d.")
        return

    # 2. Caricamento Dati
    print("   📂 Caricamento Dataset e Features...")
    df = joblib.load(d_path)
    if os.path.exists(f_path):
        X = joblib.load(f_path)
    else:
        print("   ❌ Features non trovate. Esegui prima il training!")
        return

    y = [TARGET_TO_INT[t] for t in df["target"]]

    # 3. Split (Identico al Training per coerenza)
    _, X_te, _, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # 4. DataLoader
    dl_te = DataLoader(list(zip(X_te, y_te)), batch_size=64, collate_fn=collate, shuffle=False)

    # 5. Caricamento Modello
    input_dim = X[0].shape[1]
    model = LSTMAttentionClassifier(input_dim=input_dim).to(DEVICE)
    model.load_state_dict(torch.load(m_path, map_location=DEVICE))
    model.eval()

    # 6. Inference
    print("   ⚡ Esecuzione Previsioni...")
    preds, trues = [], []
    with torch.no_grad():
        for x, l, y_b in dl_te:
            out = model(x.to(DEVICE), l.to(DEVICE))
            preds.extend(out.argmax(1).cpu().tolist())
            trues.extend(y_b.cpu().tolist())

    # 7. Calcolo Metriche
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds, average="macro")

    print("\n   🏆 RISULTATI GLOBALI")
    print(f"   -------------------")
    print(f"   Accuracy : {acc*100:.2f}%")
    print(f"   F1 Macro : {f1:.4f}")

    print("\n   📑 REPORT DETTAGLIATO")
    print(classification_report(trues, preds, target_names=["ROSSO", "GIALLO", "VERDE"], digits=4))

    # 8. Matrice di Confusione (Visuale per Tesi)
    cm = confusion_matrix(trues, preds)
    cm_df = pd.DataFrame(cm, index=["True R", "True G", "True V"], columns=["Pred R", "Pred G", "Pred V"])
    print("\n   🧮 MATRICE DI CONFUSIONE:")
    print(cm_df)
    print("\n" + "-"*60)

# --- ESECUZIONE PER TUTTI I MODELLI ---
models_to_test = [15, 30, 60]

for d in models_to_test:
    evaluate_model(d)

# Ensemple

In [ ]:
import torch
import numpy as np
import joblib
import os
from torch import nn

# --- RIDEFINIZIONE CLASSE MODELLO (Necessaria per caricare i pesi) ---
class AttentionBlock(nn.Module):
    def __init__(self, h_dim):
        super().__init__()
        self.att = nn.Sequential(nn.Linear(h_dim, h_dim//2), nn.Tanh(), nn.Linear(h_dim//2, 1))
    def forward(self, x, lens):
        sc = self.att(x)
        mask = torch.zeros_like(sc, dtype=torch.bool)
        for i, l in enumerate(lens): mask[i, :l, :] = 1
        sc = sc.masked_fill(~mask, -float('inf'))
        return torch.sum(torch.softmax(sc, dim=1) * x, dim=1)

class LSTMAttentionClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.attn = AttentionBlock(hidden_dim*2)
        self.ln = nn.LayerNorm(hidden_dim*2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim*2, 128), nn.ReLU(), nn.Dropout(0.4), nn.Linear(128, num_classes))
    def forward(self, x, l):
        packed = nn.utils.rnn.pack_padded_sequence(x, l.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        unpacked, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.fc(self.ln(self.attn(unpacked, l)))

# --- CLASSE ENSEMBLE ---
class DiabetesEnsemble:
    def __init__(self, model_dir, device):
        self.models = {}
        self.device = device
        self.days_list = [15, 30, 60, 90]
        self.input_dim = 20 # Dimensione features compressa

        print("🔧 Inizializzazione Ensemble...")
        for d in self.days_list:
            path = os.path.join(model_dir, f"model_{d}d.pth")
            if os.path.exists(path):
                # Carica architettura
                m = LSTMAttentionClassifier(input_dim=self.input_dim).to(device)
                # Carica pesi
                m.load_state_dict(torch.load(path, map_location=device))
                m.eval()
                self.models[d] = m
                print(f"   ✅ Modello {d}d caricato.")
            else:
                print(f"   ⚠️ Modello {d}d NON trovato (verrà ignorato).")

    def extract_slice(self, full_sequence, days):
        """Taglia la sequenza per prendere solo gli ultimi N giorni"""
        target_minutes = days * 1440
        current_minutes = 0
        sliced_seq = []

        # Scorriamo all'indietro (dal più recente)
        for event in reversed(full_sequence):
            state, dur, val = event
            if current_minutes + dur > target_minutes:
                # Dobbiamo tagliare l'ultimo evento parzialmente
                rem_dur = target_minutes - current_minutes
                if rem_dur > 0:
                    sliced_seq.insert(0, (state, int(rem_dur), val))
                break
            else:
                sliced_seq.insert(0, event)
                current_minutes += dur
        return sliced_seq

    def predict(self, full_sequence_90d):
        """
        Prende una sequenza lunga (es. 90+ giorni) e fa votare tutti i modelli.
        """
        predictions = {}
        probs_sum = np.zeros(3)
        active_models = 0

        with torch.no_grad():
            for days, model in self.models.items():
                # 1. Slice temporale (es. prendo solo ultimi 15gg)
                sub_seq = self.extract_slice(full_sequence_90d, days)

                if not sub_seq: continue # Skip se vuota

                # 2. Feature Engineering
                feat = build_compressed_features_v2(sub_seq) # Usa la tua funzione
                feat = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(self.device) # Batch size 1
                length = torch.tensor([len(sub_seq)]).to(self.device)

                # 3. Inference
                logits = model(feat, length)
                probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

                predictions[days] = probs

                # 4. Peso differenziato (Opzionale: puoi dare più peso al 30d/60d)
                # Qui usiamo peso uguale per tutti
                weight = 1.0
                probs_sum += probs * weight
                active_models += weight

        if active_models == 0: return None, None

        # 5. Risultato Finale (Soft Voting)
        ensemble_probs = probs_sum / active_models
        ensemble_class = np.argmax(ensemble_probs)

        return ensemble_class, ensemble_probs, predictions

print("✅ Classe Ensemble Pronta.")

# Ground truth

In [ ]:
# Mappatura Output
INT_TO_TARGET = {0: "🔴 ROSSO", 1: "🟡 GIALLO", 2: "🟢 VERDE"}

def print_diagnosis(name, seq_len, ensemble_res):
    cls_idx, probs, indiv = ensemble_res
    print(f"\n📋 PAZIENTE: {name}")
    print(f"   ⏱ Durata Dati: {seq_len/1440:.1f} giorni")
    print("-" * 60)
    print(f"   🧠 ENSEMBLE DIAGNOSIS: {INT_TO_TARGET[cls_idx]}  (Confidenza: {probs[cls_idx]*100:.1f}%)")
    print("-" * 60)
    print("   🔍 Analisi Singoli Modelli:")
    for d in [15, 30, 60, 90]:
        if d in indiv:
            p = indiv[d]
            best = np.argmax(p)
            print(f"      🗓 Modello {d}d: {INT_TO_TARGET[best]} \t[R:{p[0]:.2f} G:{p[1]:.2f} V:{p[2]:.2f}]")
        else:
            print(f"      🗓 Modello {d}d: -- Non Disponibile --")
    print("=" * 60)

# --- CREAZIONE SCENARI DI STRESS ---

# Scenario 1: Il Peggioramento Improvviso
# 75 giorni perfetti (Verde) + Ultimi 15 giorni disastrosi (Rosso)
# Ci aspettiamo: 90d vede Verde/Giallo, 15d vede Rosso Pieno. Ensemble deve allarmarsi.
seq_good = simulate_random_sequence_realistic(G, 75*1440, forced_target="verde")[0]
seq_bad  = simulate_random_sequence_realistic(G, 15*1440, forced_target="rosso")[0]
patient_worsening = seq_good + seq_bad

# Scenario 2: Il Paziente "Montagne Russe" (Variabilità Estrema)
# Alterna 10gg Rosso, 10gg Verde, 10gg Rosso... per 90gg
# Ci aspettiamo: Giallo (per l'alta variabilità) o Rosso.
patient_rollercoaster = []
for _ in range(3):
    patient_rollercoaster += simulate_random_sequence_realistic(G, 10*1440, forced_target="rosso")[0]
    patient_rollercoaster += simulate_random_sequence_realistic(G, 10*1440, forced_target="verde")[0]
    patient_rollercoaster += simulate_random_sequence_realistic(G, 10*1440, forced_target="giallo")[0]

# Scenario 3: Il Falso Allarme (Solo un giorno brutto)
# 89 giorni Verdi + 1 giorno Rosso (mangiato male)
# Ci aspettiamo: Tutti Verdi, il 15d forse leggermente meno sicuro, ma l'Ensemble deve dire Verde.
seq_stable = simulate_random_sequence_realistic(G, 89*1440, forced_target="verde")[0]
seq_spike  = simulate_random_sequence_realistic(G, 1*1440, forced_target="rosso")[0]
patient_spike = seq_stable + seq_spike

# Scenario 4: Il Cronico Silenzioso
# 90 giorni sempre Giallo (non grave, ma costante).
# Ci aspettiamo: Tutti Gialli, con alta confidenza.
patient_chronic = simulate_random_sequence_realistic(G, 90*1440, forced_target="giallo")[0]


# --- ESECUZIONE TEST ---
# Inizializza Ensemble (assicurati che MODEL_DIR sia corretto)
ensemble = DiabetesEnsemble(MODEL_DIR, DEVICE)

# 1. Test Peggioramento
res = ensemble.predict(patient_worsening)
print_diagnosis("Mario (Peggioramento Acuto)", 90*1440, res)

# 2. Test Instabile
res = ensemble.predict(patient_rollercoaster)
print_diagnosis("Luigi (Instabilità Ciclica)", 90*1440, res)

# 3. Test Spike
res = ensemble.predict(patient_spike)
print_diagnosis("Anna (Cena Pesante - Falso Allarme)", 90*1440, res)

# 4. Test Cronico
res = ensemble.predict(patient_chronic)
print_diagnosis("Giovanni (Scompenso Cronico Lieve)", 90*1440, res)